# InformationTheory-02 — La distance de Hamming et les codes correcteurs

**Série InformationTheory** · Carnet 02 — Le codage de canal (la branche **théorie
des codes** du hub « Information & calcul » #27092)

Shannon a montré, dans le carnet 01, qu'il existe pour un canal bruité un **débit
fiable** (la **capacité**). Ce carnet répond à la question suivante : *comment
l'atteindre concrètement ?* La réponse est la **théorie des codes correcteurs
d'erreurs** : ajouter juste assez d'« encre » (des parités) au message pour qu'on
puisse le retrouver, même si le bruit en a retourné quelques bits. Ce carnet
exécute, en numpy et en arithmétique GF(2) (le mod 2, l'algèbre des bits), les
briques qui répondent à cette question : la distance qui mesure « à quel point
deux mots se ressemblent », la distance minimale qui décide ce qu'un code sait
faire, le code qui **détecte**, le code qui **corrige** (le code de Hamming), et
la borne qui dit qu'on ne peut pas faire mieux.


## Enjeu — du bit retourné au message retrouvé

Tout repose sur un seul fait : un bruit de canal retourne des bits, et c'est
**rare** qu'il en retourne beaucoup d'affilée. L'idée géniale des codes
correcteurs est de **répeter de l'information** (sous forme de parités) pour que
le message utile survive : on envoie *plus* de bits que nécessaire, et cette
redondance est exactement ce qui permet, à la réception, de **détecter** qu'une
erreur est survenue, puis de la **corriger** sans re-demander le message. Tout se
jauge avec une distance — la **distance de Hamming** — et tout se calcule en GF(2).


## 0. La distance de Hamming

La **distance de Hamming** entre deux mots de même longueur est le **nombre de
positions où ils diffèrent**. C'est la métrique de base : elle mesure « à quel
point deux mots se ressemblent ». Plus deux mots sont proches (petite distance),
plus un peu de bruit peut les confondre.


In [1]:
import numpy as np

# La distance de Hamming : combien de positions separent deux mots de meme
# longueur ? C'est la metrique de base de la theorie des codes.

def distance_hamming(u, v):
    # Nombre de positions ou deux mots (listes de bits) differents.
    u, v = np.asarray(u), np.asarray(v)
    return int(np.sum(u != v))

# Mesure sur des mots reels (mots de 7 bits) :
a = [1, 0, 1, 1, 0, 1, 0]
b = [1, 0, 0, 1, 0, 1, 0]   # differ de a en une seule position (la 3e)
print(f"distance(a, b) = {distance_hamming(a, b)}")
assert distance_hamming(a, b) == 1, "a et b differe en une seule position"

c = [0, 0, 0, 1, 1, 1, 1]   # ecarte de a en 4 positions
print(f"distance(a, c) = {distance_hamming(a, c)}")
assert distance_hamming(a, c) == 4
print("=> la distance de Hamming compte les positions en desaccord : c'est la")
print("   'geometrie' dans laquelle un code correcteur vit.")


distance(a, b) = 1
distance(a, c) = 4
=> la distance de Hamming compte les positions en desaccord : c'est la
   'geometrie' dans laquelle un code correcteur vit.


### Lecture du résultat

`a` et `b` ne diffèrent qu'**en une position** (distance 1) : un seul bit retourné
les transforme l'un en l'autre — on ne pourrait **jamais** les utiliser ensemble
dans un code qui corrige. `a` et `c` diffèrent en **quatre** positions : il
faudrait quatre erreurs simultanées pour les confondre. La distance de Hamming est
donc le « thermomètre de sécurité » d'un code.


## 1. La distance minimale — ce qu'elle achète

La **distance minimale** `d` d'un code est le plus petit écart (distance de
Hamming) entre **deux mots du code**. C'est elle qui décide, par une règle
simple, ce que le code **sait faire** :

- un code de distance `d` **détecte** jusqu'à `d − 1` erreurs ;
- un code de distance `d` **corrige** jusqu'à `⌊(d − 1) / 2⌋` erreurs.


In [2]:
# La distance MINIMALE d'un code : le plus petit ecart entre deux mots du code.
# Elle decide ce que le code SAIT faire :
#   un code de distance d DETECTE jusqu'a (d-1) erreurs,
#   et CORRIGE jusqu'a floor((d-1)/2) erreurs.

# Le code de repetition de longueur 3 : {000, 111}.
code_rep = [ [0, 0, 0], [1, 1, 1] ]

def distance_minimale(code):
    d = None
    for i in range(len(code)):
        for j in range(i + 1, len(code)):
            e = distance_hamming(code[i], code[j])
            d = e if d is None else min(d, e)
    return d

d_min = distance_minimale(code_rep)
print(f"Code de repetition {{000, 111}} : distance minimale = {d_min}")
print(f"  -> detecte jusqu'a {d_min - 1} erreur(s)")
print(f"  -> corrige jusqu'a {(d_min - 1) // 2} erreur(s)")
assert d_min == 3
print("=> la distance minimale est la 'reserve de securite' du code :")
print("   plus elle est grande, plus le code resiste au bruit.")


Code de repetition {000, 111} : distance minimale = 3
  -> detecte jusqu'a 2 erreur(s)
  -> corrige jusqu'a 1 erreur(s)
=> la distance minimale est la 'reserve de securite' du code :
   plus elle est grande, plus le code resiste au bruit.


### Lecture du résultat

Le code de répétition `{000, 111}` a distance minimale **3** : il faut retourner
3 bits pour transformer `000` en `111`. La règle en découle directement — il
**détecte** jusqu'à 2 erreurs, et **corrige** jusqu'à 1 (on vote à la majorité :
`100` → `000`, `011` → `111`). C'est le principe de tous les codes correcteurs,
déjà visible dans ce code trivial.


## 2. Le code de parité — détecter une erreur

Le **code de parité** est le plus simple des codes qui **détectent**. On ajoute
au message un bit tel que la somme de tous les bits (en GF(2)) soit **paire**.
À la réception, on somme les bits : si la parité est **impaire**, on sait qu'une
erreur (de poids impair) est survenue. Le code a distance 2 → il **détecte** 1
erreur, mais ne la **corrige pas** : on sait qu'il y a un problème, pas où.


In [3]:
# Le code de PARITE : on ajoute au message un bit tel que la somme des bits
# (en GF(2)) soit PAIRE. Le mot code a donc une parite paire.

def encoder_parite(msg):
    msg = list(msg)
    p = sum(msg) % 2            # bit de parite : somme GF(2) des bits du message
    msg.append(p)
    return msg

# Le code = tous les messages de 4 bits + leur parite (16 mots de 5 bits).
tous4 = [[(i >> 3) & 1, (i >> 2) & 1, (i >> 1) & 1, i & 1] for i in range(16)]
code_parite = [ encoder_parite(m) for m in tous4 ]

print(f"Distance minimale du code de parite (4 -> 5 bits) : {distance_minimale(code_parite)}")
assert distance_minimale(code_parite) == 2, "un bit de parite donne distance 2"

# DETECTION : a la reception, on somme les bits recus. Parite impaire = erreur.
def parite_ok(mot):
    return sum(mot) % 2 == 0

mot_envoye = encoder_parite([1, 0, 1, 1])   # = [1, 0, 1, 1, 1]
mot_recu   = mot_envoye.copy(); mot_recu[2] ^= 1   # 1 erreur sur le 3e bit
print(f"Envoye : {mot_envoye}   parite ok ? {parite_ok(mot_envoye)}")
print(f"Recu   : {mot_recu}   parite ok ? {parite_ok(mot_recu)}")
assert parite_ok(mot_envoye) and not parite_ok(mot_recu)
print("=> la parite DETECTE 1 erreur (distance 2) -- mais ne la CORRECT pas :")
print("   on sait qu'il y a un probleme, pas OU.")


Distance minimale du code de parite (4 -> 5 bits) : 2
Envoye : [1, 0, 1, 1, 1]   parite ok ? True
Recu   : [1, 0, 0, 1, 1]   parite ok ? False
=> la parite DETECTE 1 erreur (distance 2) -- mais ne la CORRECT pas :
   on sait qu'il y a un probleme, pas OU.


### Lecture du résultat

Le message `[1,0,1,1]` s'encode en `[1,0,1,1,1]` (la parité rend la somme
paire). Une **seule** erreur (le 3e bit retourné) rend la somme impaire : la
parité **craque** et signale le problème. Mais le code de parité ne dit **pas**
quel bit est faux — il a distance 2, il **détecte** sans **corriger**. Pour
corriger, il faut un code plus fort.


## 3. Le code de Hamming (7,4) — corriger par syndrome

Le **code de Hamming** est le premier code qui **corrige**. Il encode 4 bits de
message en 7 bits, a distance minimale 3, et **corrige toute erreur unitaire**
par **décodage par syndrome**. L'astuce est élégante : la matrice de contrôle `H`
a 3 lignes et 7 colonnes, et **la colonne `j` est l'écriture binaire de `j`**.
Le syndrome d'un mot reçu vaut `H · reçu` (en GF(2)) : s'il est nul, pas
d'erreur ; s'il vaut la **colonne `j`**, c'est que le bit `j` est faux. Le
syndrome **pointe** l'erreur du doigt.


In [4]:
# Le code de HAMMING (7,4) : le premier code qui CORRECT.
# Il encode 4 bits en 7, distance minimale 3, corrige TOUTE erreur unitaire
# par DECODAGE PAR SYNDROME. La colonne j de H est l'ecriture binaire de j.

H = np.array([
    [1, 0, 1, 0, 1, 0, 1],   # bit 0 des colonnes 1..7
    [0, 1, 1, 0, 0, 1, 1],   # bit 1
    [0, 0, 0, 1, 1, 1, 1],   # bit 2
], dtype=int)

def syndrome(H, mot):
    return (H @ np.asarray(mot, dtype=int)) % 2

def encoder_hamming(msg):
    # 4 bits de message -> mot code 7 bits. Les positions de parite (1,2,4)
    # forment l'identite : les parites sont H_data @ msg (en GF(2)).
    msg = np.asarray(msg, dtype=int)
    H_data = H[:, [2, 4, 5, 6]]          # colonnes 3,5,6,7 (les 4 bits de message)
    parites = (H_data @ msg) % 2          # 3 bits de parite (positions 1,2,4)
    mot = np.zeros(7, dtype=int)
    mot[[0, 1, 3]] = parites              # positions 1,2,4
    mot[[2, 4, 5, 6]] = msg               # positions 3,5,6,7
    return mot

# VERIFICATION : chaque mot code a un syndrome nul.
for msg in ([0, 0, 0, 0], [1, 1, 1, 1], [1, 0, 1, 0]):
    assert np.all(syndrome(H, encoder_hamming(msg)) == 0), "un mot code a syndrome nul"
print("Encodage OK : les mots codes ont un syndrome nul.")

# DISTANCE MINIMALE : le code de Hamming (7,4) a distance 3.
tous4 = [[(i >> 3) & 1, (i >> 2) & 1, (i >> 1) & 1, i & 1] for i in range(16)]
code = [ encoder_hamming(m) for m in tous4 ]
d_min = distance_minimale(code)
print(f"Distance minimale du code de Hamming (7,4) : {d_min}")
assert d_min == 3, "code de Hamming : distance 3 (corrige 1 erreur)"

# CORRECTION PAR SYNDROME : une erreur au bit j donne le syndrome = colonne j.
def decoder_hamming(H, recu):
    s = syndrome(H, recu)
    if np.all(s == 0):
        return recu, None                  # pas d'erreur (detectee)
    for j in range(len(recu)):
        if np.all(H[:, j] == s):           # syndrome = colonne j
            corrigee = recu.copy(); corrigee[j] ^= 1
            return corrigee, j + 1        # position 1..7
    raise ValueError("2 erreurs ou plus : non corrigeable (distance 3)")

msg_envoye = [1, 0, 1, 1]
mot  = encoder_hamming(msg_envoye)
recu = mot.copy(); recu[4] ^= 1           # 1 erreur sur le bit 5
corrige, pos = decoder_hamming(H, recu)
print(f"Envoye : {mot.tolist()}")
print(f"Recu   : {recu.tolist()}   (erreur injectee au bit 5)")
print(f"Corrige: {corrige.tolist()}   (syndrome -> bit {pos})")
assert np.all(corrige == mot) and pos == 5, "l'erreur unitaire est localisee et corrige"
print("=> le syndrome pointe EXACTEMENT le bit en erreur : 1 erreur, corrigee.")


Encodage OK : les mots codes ont un syndrome nul.
Distance minimale du code de Hamming (7,4) : 3
Envoye : [0, 1, 1, 0, 0, 1, 1]
Recu   : [0, 1, 1, 0, 1, 1, 1]   (erreur injectee au bit 5)
Corrige: [0, 1, 1, 0, 0, 1, 1]   (syndrome -> bit 5)
=> le syndrome pointe EXACTEMENT le bit en erreur : 1 erreur, corrigee.


### Lecture du résultat

Le message `[1,0,1,1]` s'encode en un mot de 7 bits au syndrome nul. Une erreur
sur le **bit 5** est injectée : le syndrome du mot reçu vaut alors **la colonne 5**
de `H`, et le décodeur, en comparant le syndrome à chaque colonne, **identifie la
position 5** et retourne le bit. Le mot d'origine est **reconstruit exactement**.
C'est la différence avec la parité : le code de Hamming ne se contente pas de
signaler « il y a un problème », il **dit où** et **répare**.


## 4. La borne de Hamming — pourquoi « parfait »

On ne peut pas corriger indéfiniment : chaque mot code « réclame » autour de lui
une **sphère de rayon 1** — lui-même, plus les `n` mots à distance 1 (une erreur
sur chacun des `n` bits), soit `1 + n` mots. Pour que le code **corrige**, ces
sphères doivent être **disjointes** (sinon un mot reçu pourrait venir de deux
mots codes différents). D'où la **borne de Hamming** :

$$2^{k} \cdot (1 + n) \;\leq\; 2^{n}$$

Le code de Hamming (7,4) **saturé** cette borne : c'est un code **parfait**.


In [5]:
# La BORNE DE HAMMING (borne de pavage de spheres) : un code qui corrige 1
# erreur, de longueur n, a ses spheres de rayon 1 DISJOINTES, donc :
#     2^k * (1 + n)  <=  2^n
# Le code de Hamming (7,4) SATT la borne : il est PARFAIT.

n, k = 7, 4
volume_sphere = 1 + n               # 1 (le mot) + n (une erreur par bit)
nb_mots       = 2 ** k
cote_gauche   = nb_mots * volume_sphere
cote_droite   = 2 ** n
print(f"Code de Hamming (n={n}, k={k}) :")
print(f"  {nb_mots} mots x {volume_sphere} mots/sphere = {cote_gauche}")
print(f"  taille de l'espace 2^{n}            = {cote_droite}")
assert cote_gauche == cote_droite, "le code de Hamming est PARFAIT : borne atteinte"
print("=> 16 x 8 = 128 = 2^7 : les spheres de rayon 1 pavent EXACTEMENT")
print("   l'espace. C'est le sens du mot 'parfait' : aucune place gaspillee.")


Code de Hamming (n=7, k=4) :
  16 mots x 8 mots/sphere = 128
  taille de l'espace 2^7            = 128
=> 16 x 8 = 128 = 2^7 : les spheres de rayon 1 pavent EXACTEMENT
   l'espace. C'est le sens du mot 'parfait' : aucune place gaspillee.


### Lecture du résultat

Les 16 mots du code de Hamming, chacun avec sa sphère de 8 mots (lui + les 7
à une erreur), couvrent **exactement** les $2^7 = 128$ mots possibles de 7 bits
— ni plus, ni moins. C'est ce que « **parfait** » veut dire : le code utilise
**tout** l'espace disponible pour la correction, sans gaspillage. C'est une
propriété rare et précieuse, et elle explique pourquoi le (7,4) est le code
correcteur de référence.


## Synthèse — de la distance à la correction

| Brique | Ce qu'on a montré |
|---|---|
| Distance de Hamming | `a`,`b` diffèrent en 1 bit → distance 1 ; `a`,`c` en 4 bits → distance 4 |
| Distance minimale | le code `{000,111}` a `d = 3` → détecte 2, corrige 1 |
| Code de parité | distance 2 → **détecte** 1 erreur (parité impaire), ne corrige pas |
| Code de Hamming (7,4) | distance 3 → **corrige** 1 erreur ; le **syndrome** pointe le bit fautif |
| Borne de Hamming | 16 × 8 = 128 = 2⁷ → le (7,4) est **parfait** (borne saturée) |

Le fil rouge : **mesurer l'écart (distance), en faire une réserve (distance
minimale), la dépenser à détecter puis à corriger, et ne jamais dépasser la borne.**


## Exercices


In [6]:
# Exercice 1 — Pourquoi d - 1 ?
# Prouver qu'un code de distance minimale d DETECTE toutes les erreurs de poids
# <= d-1 (mais pas necessairement d) : raisonner par l'inegalite triangulaire
# de la distance de Hamming (un mot recu a 1 erreur d'un mot code ne peut
# coincider qu'avec lui si la distance entre mots codes est > 1).
def exercice_pourquoi_d_moins_1():
    # TODO etudiant
    return None  # TODO etudiant


In [7]:
# Exercice 2 — La parite face a 2 erreurs
# Le code de parite (distance 2) detecte 1 erreur. DETECTE-t-il 2 erreurs ?
# Non : 2 bits retournes conservent la parite paire. Montrer par un exemple
# concret : encoder un message, retourner 2 bits, verifier que parite_ok
# renvoie True (l'erreur passe inapercue) -- d'ou la need d'un code de
# distance 3 pour corriger.
def exercice_parite_2_erreurs():
    # TODO etudiant
    return None  # TODO etudiant


In [8]:
# Exercice 3 — La famille des codes de Hamming
# Le code de Hamming general a n = 2^m - 1 bits, k = 2^m - m - 1 bits de
# message. Pour m = 4 : n = 15, k = 11. Construire sa matrice H (4 lignes,
# colonnes = binaire de 1..15), verifier la distance 3, et montrer que la borne
# de Hamming est SATTE (2^11 x 16 = 2^15) : le code (15,11) est aussi PARFAIT.
def exercice_hamming_15_11():
    # TODO etudiant
    return None  # TODO etudiant


## À retenir

- **La distance de Hamming** compte les positions où deux bits diffèrent — c'est
  la géométrie d'un code.
- **La distance minimale** `d` décide tout : un code **détecte** `d − 1` erreurs
  et **corrige** `⌊(d − 1)/2⌋`.
- **Le code de parité** (distance 2) **détecte** 1 erreur mais ne la corrige pas.
- **Le code de Hamming (7,4)** (distance 3) **corrige** 1 erreur : le **syndrome**
  égale la colonne de `H` du bit fautif, il **pointe** l'erreur.
- **La borne de Hamming** `2^k (1+n) ≤ 2^n` : le (7,4) la **sature** — c'est un
  code **parfait**, 16 × 8 = 128 = 2⁷.


## Bibliographie

- W. W. Peterson, D. J. Weldon, *Error-Correcting Codes*, 3ᵉ éd., MIT Press,
  1997 — le référentiel de la théorie des codes correcteurs.
- R. G. Gallager, *Low-Density Parity-Check Codes*, MIT Press, 1963 — les
  codes qui, avec Hamming, fondent la correction moderne.
- Carnet 03 (à venir) : les codes cycliques et Reed-Solomon — le générateur
  polynômial, les codes BCH, et pourquoi ils sont partout (CD, QR, 5G).
